# COMPSCI764 Colab Runner

Orchestration only. Do not execute during Phase 1A. Phase 1B requires separate authorisation, an exact reviewed commit, a reviewed dependency lock, and an explicitly approved candidate profile. All blank inputs below must be resolved before execution. No profile becomes tested until Run A and a fresh-runtime Run B pass and the resulting lock/profile are reviewed and committed. Never push or edit project code from Colab.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/DanielS26/rag-cad-semantic-pairs.git"
COMMIT = ""
BOOTSTRAP_SHA256 = ""
PHASE1B_AUTHORISED = False
ALLOW_NETWORK = False
STAGE = "run-a"
RUN_ID = ""
SOURCE_MANIFEST = ""
RUN_A_MANIFEST = ""
CONNECTED_SECONDS_USED = None
COMPUTE_UNITS_USED = None
COMPUTE_UNITS_PER_HOUR = None
CHECKOUT = Path("/content/rag-cad-semantic-pairs")
WORK = Path("/content/work")
PROFILE_RELATIVE = "configs/environments/colab_candidate.yaml"
LOCK_RELATIVE = "requirements/colab-core.candidate.lock.txt"
STORAGE_RELATIVE = "configs/storage.yaml"
READY = False
STAGE_COMPLETE = False

In [ ]:
import hashlib
import math
import os
import re
import runpy
import subprocess
import tempfile
import time

if not PHASE1B_AUTHORISED or not RUN_ID or ALLOW_NETWORK is not True:
    raise RuntimeError("Separate Phase 1B and bootstrap-network permission and a new run ID are required")
if not re.fullmatch(r"[0-9a-f]{40}", COMMIT) or not re.fullmatch(r"[0-9a-f]{64}", BOOTSTRAP_SHA256):
    raise RuntimeError("Supply the exact reviewed commit and bootstrap file hash")
if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+\.git", REPO_URL):
    raise RuntimeError("Expected a clean GitHub HTTPS URL")
if any(type(value) not in (int, float) or not math.isfinite(value) or value < 0
       for value in (CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR)):
    raise RuntimeError("Enter actual cumulative connected time, consumed units, and the displayed unit rate")
if COMPUTE_UNITS_PER_HOUR <= 0 or CONNECTED_SECONDS_USED >= 10800 or COMPUTE_UNITS_USED >= 20:
    raise RuntimeError("Budget unavailable or exhausted; disconnect and delete runtime")
STARTED = time.monotonic()
BOOTSTRAP_TIMEOUT = min(10800 - CONNECTED_SECONDS_USED,
                        (20 - COMPUTE_UNITS_USED) * 3600 / COMPUTE_UNITS_PER_HOUR)
if CHECKOUT.exists():
    raise RuntimeError("Use a fresh runtime checkout")
from google.colab import userdata

clone_env = os.environ.copy()
for name in ("GH_TOKEN", "GITHUB_TOKEN", "GH_READ_TOKEN"):
    clone_env.pop(name, None)
clone_failure = None
try:
    with tempfile.TemporaryDirectory(prefix="rag-cad-askpass-") as temporary:
        askpass = Path(temporary) / "askpass.sh"
        askpass.write_text('#!/bin/sh\ncase "$1" in\n*Username*) printf "%s\\n" "x-access-token" ;;\n*Password*) printf "%s\\n" "$GH_READ_TOKEN" ;;\n*) exit 1 ;;\nesac\n', encoding="utf-8")
        askpass.chmod(0o700)
        clone_env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0",
                         GH_READ_TOKEN=userdata.get("GH_READ_TOKEN"))
        subprocess.run(["git", "-c", "credential.helper=", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)],
                       env=clone_env, check=True, capture_output=True, text=True, timeout=BOOTSTRAP_TIMEOUT)
except subprocess.TimeoutExpired:
    clone_failure = "timed_out"
except subprocess.CalledProcessError as error:
    clone_failure = f"exit_code={error.returncode}" if type(error.returncode) is int else "failed"
except Exception:
    clone_failure = "failed"
finally:
    clone_env.clear()
    del clone_env
if clone_failure is not None:
    raise RuntimeError(f"git_clone: {clone_failure}") from None
checkout_failure = None
checkout_stage = "git_checkout"
try:
    subprocess.run(["git", "checkout", "--detach", COMMIT], cwd=CHECKOUT, check=True, capture_output=True, text=True, timeout=120)
    checkout_stage = "checkout_verification"
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=CHECKOUT, text=True, stderr=subprocess.PIPE).strip()
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=CHECKOUT, text=True, stderr=subprocess.PIPE).strip()
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=CHECKOUT, text=True, stderr=subprocess.PIPE).strip()
except subprocess.TimeoutExpired:
    checkout_failure = "timed_out"
except subprocess.CalledProcessError as error:
    checkout_failure = f"exit_code={error.returncode}" if type(error.returncode) is int else "failed"
except Exception:
    checkout_failure = "failed"
if checkout_failure is not None:
    raise RuntimeError(f"{checkout_stage}: {checkout_failure}") from None
if actual != COMMIT or remote != REPO_URL or dirty:
    raise RuntimeError("Exact clean checkout verification failed")
bootstrap_path = CHECKOUT / "scripts/bootstrap_colab.py"
if hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != BOOTSTRAP_SHA256:
    raise RuntimeError("Local bootstrap integrity check failed")
bootstrap = runpy.run_path(str(bootstrap_path), run_name="trusted_bootstrap")
bootstrap["verify_checkout"](CHECKOUT, REPO_URL, COMMIT)

In [ ]:
READY = False
if not PHASE1B_AUTHORISED or ALLOW_NETWORK is not True:
    raise RuntimeError("Dependency installation requires separately approved network permission")
remaining = bootstrap["remaining_seconds"](STARTED, CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR)
PYTHON = bootstrap["install_and_test"](CHECKOUT, REPO_URL, COMMIT, LOCK_RELATIVE, allow_network=ALLOW_NETWORK, timeout=remaining)
WORK.mkdir(parents=True, exist_ok=True)
common = [str(PYTHON), "-m", "rag_cad.cli", "--repo", str(CHECKOUT)]
gate = ["--profile", str(CHECKOUT / PROFILE_RELATIVE), "--lock", str(CHECKOUT / LOCK_RELATIVE),
        "--commit", COMMIT, "--workspace", str(WORK), "--execute"]
bootstrap["run_stage"](common + ["environment", "preflight", "--stage", "smoke", *gate,
                        "--output", str(WORK / "preflight.json")], stage="environment_preflight", cwd=CHECKOUT,
               timeout=bootstrap["remaining_seconds"](STARTED, CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR))
READY = True

In [ ]:
if not READY or not PHASE1B_AUTHORISED:
    raise RuntimeError("Verify checkout and pass preflight before artefact operations")
bootstrap["verify_checkout"](CHECKOUT, REPO_URL, COMMIT)
if STAGE == "run-b" and not RUN_A_MANIFEST:
    raise RuntimeError("Run B requires the published Run A manifest")
if SOURCE_MANIFEST or STAGE == "run-b":
    from google.colab import drive
    drive.mount("/content/drive")
    try:
        if SOURCE_MANIFEST:
            bootstrap["run_stage"](common + ["artefact", "restore", "--manifest", SOURCE_MANIFEST,
                "--destination", str(WORK / "inputs")], stage="artefact_restore", cwd=CHECKOUT,
                timeout=bootstrap["remaining_seconds"](STARTED, CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR))
        if STAGE == "run-b":
            bootstrap["run_stage"](common + ["artefact", "restore", "--manifest", RUN_A_MANIFEST,
                "--destination", str(WORK / "restored_run_a"), "--receipt", str(WORK / "run_a_receipt.json")],
                stage="artefact_restore", cwd=CHECKOUT,
                timeout=bootstrap["remaining_seconds"](STARTED, CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR))
    finally:
        drive.flush_and_unmount()
source = WORK / "inputs" / "upstream"
resources = WORK / "inputs" / "nltk_data"

In [ ]:
STAGE_COMPLETE = False
if not READY or not PHASE1B_AUTHORISED or STAGE not in {"run-a", "run-b"}:
    raise RuntimeError("Only an authorised bounded smoke stage is allowed")
if os.path.ismount("/content/drive"):
    raise RuntimeError("Unmount Drive before model loading or smoke execution")
if not source.exists():
    if not ALLOW_NETWORK:
        raise RuntimeError("Restore verified inputs or explicitly authorise named downloads")
    bootstrap["run_stage"](common + ["upstream", "fetch-cad", *gate,
        "--release", str(CHECKOUT / "configs/upstream/cad_release.yaml"),
        "--source", str(source), "--output", str(WORK / "source_manifest.json"),
        "--allow-network"], stage="upstream_fetch", cwd=CHECKOUT,
        timeout=bootstrap["remaining_seconds"](STARTED, CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR))
stage_output = WORK / "outputs" / RUN_ID
stage_command = common + ["smoke", STAGE, *gate,
    "--release", str(CHECKOUT / "configs/upstream/cad_release.yaml"),
    "--source", str(source), "--resources", str(resources),
    "--model-bundle", str(WORK / "inputs" / "jina-bundle")]
if STAGE == "run-a":
    stage_command += ["--output", str(stage_output)]
else:
    stage_command += ["--receipt", str(WORK / "run_a_receipt.json"), "--destination", str(WORK / "restored_run_a"),
                      "--output", str(stage_output / "run_b.json")]
if ALLOW_NETWORK:
    stage_command.append("--allow-network")
try:
    bootstrap["run_stage"](stage_command, stage="smoke_run_a" if STAGE == "run-a" else "smoke_run_b", cwd=CHECKOUT,
                   timeout=bootstrap["remaining_seconds"](STARTED, CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR))
finally:
    for diagnostic in ("smoke_progress.json", "last_failure.json"):
        if (WORK / diagnostic).is_file():
            print(diagnostic, (WORK / diagnostic).read_text(encoding="utf-8").strip())
STAGE_COMPLETE = True

In [ ]:
if not READY or not PHASE1B_AUTHORISED or not STAGE_COMPLETE:
    raise RuntimeError("Publication requires a completed authorised stage")
remaining = bootstrap["remaining_seconds"](STARTED, CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR)
from google.colab import drive

drive.mount("/content/drive")
try:
    bootstrap["run_stage"](common + ["artefact", "publish", "--source", str(stage_output),
        "--stage", "smoke", "--run-id", RUN_ID,
        "--storage-config", str(CHECKOUT / STORAGE_RELATIVE),
        "--metadata", str(WORK / "preflight.json")], stage="artefact_publish", cwd=CHECKOUT,
        timeout=bootstrap["remaining_seconds"](STARTED, CONNECTED_SECONDS_USED, COMPUTE_UNITS_USED, COMPUTE_UNITS_PER_HOUR))
finally:
    drive.flush_and_unmount()
print({"commit": COMMIT, "run_id": RUN_ID, "published_and_verified": True,
       "next_action": "Record actual compute units/time, then disconnect and delete runtime"})